# Completeness Comparison: transaction.csv vs data_normalise.csv

This notebook compares null counts per column between:
- `transaction.csv`
- `data_normalise.csv`

Null definition:
- real missing (`NaN`)
- empty string after trim
- literal text `NULL` (case-insensitive)


In [2]:
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

transaction_df = pd.read_csv('transaction.csv')
normalised_df = pd.read_csv('data_normalise.csv', keep_default_na=False)

print(f'transaction.csv shape: {transaction_df.shape}')
print(f'data_normalise.csv shape: {normalised_df.shape}')


transaction.csv shape: (10103, 6)
data_normalise.csv shape: (10103, 7)


In [3]:
def is_null_like(value):
    if pd.isna(value):
        return True
    text = str(value).strip()
    return text == '' or text.upper() == 'NULL'

def build_null_table(df, dataset_name):
    total_rows = len(df)
    rows = []
    for col in df.columns:
        null_count = int(df[col].map(is_null_like).sum())
        rows.append({
            'dataset': dataset_name,
            'column': col,
            'total_rows': total_rows,
            'null_count': null_count,
            'null_pct': round((null_count / total_rows) * 100, 2),
        })
    return pd.DataFrame(rows).sort_values(by=['null_count', 'column'], ascending=[False, True]).reset_index(drop=True)

transaction_nulls = build_null_table(transaction_df, 'transaction.csv')
normalised_nulls = build_null_table(normalised_df, 'data_normalise.csv')

display(pd.concat([transaction_nulls, normalised_nulls], ignore_index=True))


,dataset,column,total_rows,null_count,null_pct
0,transaction.csv,card_type,10103,314,3.11
1,transaction.csv,city,10103,117,1.16
2,transaction.csv,time,10103,8,0.08
3,transaction.csv,amount,10103,2,0.02
4,transaction.csv,id,10103,0,0.00
5,transaction.csv,status,10103,0,0.00
6,data_normalise.csv,card_id,10103,314,3.11
7,data_normalise.csv,amount,10103,301,2.98
8,data_normalise.csv,city_id,10103,117,1.16
9,data_normalise.csv,date,10103,23,0.23


In [4]:
column_map = {
    'status': 'status',
    'time': 'time',
    'amount': 'amount',
    'id': 'transaction_id',
    'card_type': 'card_id',
    'city': 'city_id',
}

comparison_rows = []
for src_col, norm_col in column_map.items():
    src_null = int(transaction_df[src_col].map(is_null_like).sum()) if src_col in transaction_df.columns else None
    norm_null = int(normalised_df[norm_col].map(is_null_like).sum()) if norm_col in normalised_df.columns else None
    comparison_rows.append({
        'transaction_column': src_col,
        'normalised_column': norm_col,
        'transaction_null_count': src_null,
        'normalised_null_count': norm_null,
        'null_count_change': None if (src_null is None or norm_null is None) else (norm_null - src_null),
    })

display(pd.DataFrame(comparison_rows))


,transaction_column,normalised_column,transaction_null_count,normalised_null_count,null_count_change
0,status,status,0,0,0
1,time,time,8,23,15
2,amount,amount,2,301,299
3,id,transaction_id,0,0,0
4,card_type,card_id,314,314,0
5,city,city_id,117,117,0
